# Variational Autoencoders (VAEs)

**Variational Autoencoders (VAEs)** are a class of generative models that learn to compress data into a compact, structured latent space and then reconstruct it back. Training a VAE involves two coupled objectives. The encoder learns to map observations $x$ into a distribution over latent codes $q_\lambda(z|x)$, summarizing what the model believes about the hidden causes of a given input. The decoder then learns to reverse this process, reconstructing $x$ from samples of $z$ via $p(x \mid z, \phi)$. Because the true posterior over $z$ is intractable, VAEs use variational inference to approximate it — optimizing a tractable lower bound on the ELBO instead.

It might be good to hone in on what "variational" means here exactly. The word "variational" comes from **variational inference**, which is a technique for approximating intractable posteriors. Recall the central issue we've run into so far: the likelihood
$$p(x|\phi) = \int p(x, z|\phi)dz = \int p(x|z,\phi) \cdot p(z) dz $$
requires integrating out $z$ -- something we can't do. Equivalently, computing the posterior via Baye's rule
$$p(z\mid x, \phi) = \frac{p(x\mid z, \phi) p(z)}{p(x\mid \phi)}$$
runs into the same problem in the denominator. 

The variational idea is to approximate this intractable posterior with a simpler, tractable distribution $q(z\mid \theta)$, which is an idea that we've already seen in the discussion about the ELBO. To that end, "variational" refers to the fact that instead of computing the true posterior exactly, we **vary** the parameters $\theta$ of an approximate distribution to make it as close to the true posterior as possible. In this case, we choose a multivariate Gaussian distribution with mean $\mu$ and diagonal variance $\Sigma$. 

Note that this will not always match the posterior well but will be better for some values than others. Also note that the use of a shared inference network (e.g. neural network) to map observations directly to posterior parameters is known as **amortized variational inference**.

## Building the model

To build the model, recall the ELBO from the reconstruction error perspective:
$$ \text{ELBO}(\theta, \phi) = \int q(z|\theta) \log(p(x|z, \theta))dz + D_{\text{KL}}\left[p(z)|q(z|\theta)\right]$$
where we make the approximation of the distribution $q(z|\theta)$. Since the optimal choice for the approximation is $p(z|x)$ (makes the lower bound tight), we will go with
$$q(z|\theta) = \mathcal{N}(g_\mu(x, \theta), g_\Sigma(x, \theta))$$
where $g(x, \theta)$ is a second neural network (the first one $f(\cdot)$ was for prediction) that predicts the mean and variance of the normal variational approximation. 

Going back to the ELBO, we know that the first time is still an intractable integral. However, since it is also the expectation of the log probability with respect to $q(z|\theta)$, we can approximate it by sampling. More specifically, we can get a **Monte Carlo estimate** where given a function $a(\cdot)$,
$$\mathbb{E}_z[a(z)] = \int a(z) q(z|\theta) dz \approx \frac{1}{N}\sum^N_{n=1}a(z^*_n)$$
where $z^*_n$ is the $n$-th sample from $q(z|\theta)$. For a very approximate estimate, we can just use a single sample $z^*$. This yields an updated ELBO approximation:
$$\text{ELBO}(\theta, \phi) \approx \log(p(x|z^*, \theta)) + D_{\text{KL}}\left[p(z)|q(z|\theta)\right]$$

For the KL-divergence, we can obtain an exact form as both $q(z|\theta)$ and $p(z)$ are approximated as normal distributions:
$$D_{\text{KL}}[q(z|\theta) | p(z)] = \frac{1}{2}\left(\text{Tr}(\Sigma) +\mu^\top \mu - D_z -\log\left[\text{det}(\Sigma)\right]\right)$$
where $D_z$ is the dimensionality of the latent space.

There's a lot going on here, so let's summarize everything. Our main objective is to figure out a way to calculate the ELBO for a point $x$ as it's a tractable loss function that can be calculated. That is, once we have the ELBO, everything else falls into place. 
1. Compute the mean $\mu$ and variance $\Sigma$ of the variational normal approximation for the posterior $q(z|\theta,x)$ using the neural network $g(x, \theta)$.
2. Draw a sample $z^*$ from the distribution (using reparameterization trick as we will see below).
3. Compute the mean of the conditional likelihood $p(x | z^*, \phi)$ using the neural network $f(z^*, \phi)$.

Hopefully, the term **autoencoder** is starting to make more intuitive sense now. Starting with a datapoint $x$, we compute the lower dimensional latent representation $z$, and use this latent variable to recreate the original datapoint as closely as possible. In this case, the encoder refers to the neural network $g(x, \theta)$ and the decoder refers to the network $f(z^*, \phi)$. 

We will talk in more detail about the algorithm in a second, but it's important to highlight the role of the ELBO throughout this process. Like many other ML training schemes, the idea is to maximize the loss function via backpropagation and SGD. The ELBO loss is a elegant way to jointly train both neural networks, as it encapsulates both the encoder and decoder portion. The log likelihood $\log p(x | z^*, \phi)$ corresponds to reconstruction quality (decoder) and the the KL-divergence penalizes the variational distribution from straying too far from the prior (encoder).

![VAE diagram](VAE.png)


## Reparameterization 
There is one more small technical detail. During the training process, we must sample a latent data point $z^*$. However, how exactly do we differentiate this stochastic component during backpropagation? Suppose for a second that we are using the Monte Carlo estimate for the conditional log-likelihood:
$$\frac{1}{N} \sum^N_{n=1}\log p(x_n | z^*_n, \phi)$$
It follows that we can easily obtain a gradient with respect to $\phi$ as $p$ is approximated with a Gaussian. But the gradient with respect to $\theta$ is not quite as simple because changes to $\theta$ will change the distribution of $q(z_n | x, \theta)$, which in turn changes the sample $z^*$, a fixed value.

We resolve this using a simple **reparameterization trick** in which we reformulate the Monte Carlo sampling procedure such that we can easily calculate the gradient wrt to $\theta$. First, note that if $\epsilon$ is a Gaussian random variable with zero mean and unit variance, then the quantity
$$z = \mu + \sigma \epsilon$$
will be a Gaussian centered at $\mu$ with variance $\sigma^2$. So, instead of drawing samples $z^*_n$ directly, we draw samples from $\epsilon$ and calculate corresponding samples for
$$z^*_n = g_\mu(x, \theta) + g_\Sigma^{\frac{1}{2}}(x, \theta)\epsilon^*_n$$ 
Now, the randomness is isolated in the noise term $\epsilon^*_n$, which does not depend on $\theta$. This means that the gradient becomes
$$\nabla_\theta z_n^* = \nabla_\theta g_\mu(x, \theta) + \nabla_\theta g_\Sigma^{\frac{1}{2}}(x, \theta)\epsilon^*_n$$
which is well-defined and amenable to backprop.